In [ ]:
import pandas as pd
import numpy as np

# Indlæs data
sales = pd.read_csv("Sales.csv")
merged = pd.read_csv("merged_bricks_clean_full.csv")

# Standardisér matchnøgler for at undgå whitespace-case issues
sales['__key']  = sales['MUNICIPALITY'].astype(str).str.strip()
merged['__key'] = merged['brick'].astype(str).str.strip()

# Merge: MUNICIPALITY (sales) vs brick (merged) via de standardiserede nøgler
sales_with_brick = sales.merge(
    merged[['__key', 'brick_nr']],
    left_on="__key",
    right_on="__key",
    how="left"
)

# Ryd op i hjælpekolonnen
sales_with_brick = sales_with_brick.drop(columns=['__key'])

# --- Rens brick_nr ---
# Til string for at kunne fjerne '.0' og håndtere tekstlige "ikke-værdier"
sales_with_brick['brick_nr'] = sales_with_brick['brick_nr'].astype(str).str.strip()

# Fjern trailing '.0'
sales_with_brick['brick_nr'] = sales_with_brick['brick_nr'].str.replace('.0', '', regex=False)

# Erstat almindelige ikke-værdier med NaN
sales_with_brick['brick_nr'] = sales_with_brick['brick_nr'].replace(
    {'': np.nan, 'nan': np.nan, 'NaN': np.nan, 'None': np.nan, 'NULL': np.nan}
)

# Sæt København = 100 (case-insensitivt og uden leading/trailing spaces)
is_kbh = sales_with_brick['MUNICIPALITY'].astype(str).str.strip().str.casefold() == 'københavn'
sales_with_brick.loc[is_kbh, 'brick_nr'] = '100'

# Tving til numerisk (alt ikke-numerisk bliver NaN)
sales_with_brick['brick_nr'] = pd.to_numeric(sales_with_brick['brick_nr'], errors='coerce')

# Drop rækker uden gyldigt brick_nr
sales_with_brick = sales_with_brick.dropna(subset=['brick_nr'])

# Konverter til int
sales_with_brick['brick_nr'] = sales_with_brick['brick_nr'].astype(int)

# Gem resultat
sales_with_brick.to_csv("Sales_with_brick.csv", index=False)

# Hurtig sanity check
print(sales_with_brick[['MUNICIPALITY', 'brick_nr']].head())
print("Antal rækker efter rens:", len(sales_with_brick))


   MUNICIPALITY  brick_nr
1      Hillerød       111
2     Haderslev       311
10      Randers       411
12       Assens       303
13    København       100
Antal rækker efter rens: 108317


In [6]:
import pandas as pd
from collections import OrderedDict

# --- Load data ---
df = pd.read_csv("Sales_with_brick.csv")

# --- Helper to format integers with dot as thousands separator (e.g., 1.374) ---
def fmt(n: int) -> str:
    return f"{int(n):,}".replace(",", ".")

# --- Optional: ATC code -> human name mapping (fill in if you have it) ---
ATC_NAME_MAP = {
    # Example:
    # "A10AB05": "Insulin Aspart",
    # Add more if you have a mapping
}

# ==============================
# Drug Type Codes (WHO_ATC_CODE)
# ==============================
code_counts = df["WHO_ATC_CODE"].value_counts(dropna=False)

n_unique_codes = df["WHO_ATC_CODE"].nunique(dropna=False)

# Top 5 most used codes
top5 = code_counts.head(5)

# Most common & least common codes
most_common_code = code_counts.idxmax()
most_common_count = code_counts.max()

least_common_code = code_counts.idxmin()
least_common_count = code_counts.min()

top5_list = []
for i, (code, cnt) in enumerate(top5.items(), start=1):
    top5_list.append(f"{i}.  {code} ({fmt(cnt)})")
top5_str = " ".join(top5_list)

print(f"There are {n_unique_codes} unique Drug Type Codes present in the data, "
      f"with the 5 most used codes being: {top5_str}")

# Second line: show most and least common 
most_common_name = ATC_NAME_MAP.get(most_common_code)
least_common_name = ATC_NAME_MAP.get(least_common_code)

print(f"\nThere are {n_unique_codes} unique Drug Type Codes present in the dataset. "
      f"The most common Drug Type Code in the dataset is {most_common_code} "
      f"(“{most_common_name}”), present in {fmt(most_common_count)} rows/sales.  "
      f"The least common Drug Type Code in the dataset is {least_common_code} "
      f"(“{least_common_name}”), present in {fmt(least_common_count)} rows/sales.")

# ==============================
# Municipalities
# ==============================
n_munis = df["MUNICIPALITY"].nunique()
print(f"\nThere are {n_munis} unique municipalities in the dataset.")

# Top 5 municipalities by total VALUE and by total VOLUME
muni_value = (
    df.groupby("MUNICIPALITY", as_index=False)["VALUE"]
      .sum()
      .sort_values("VALUE", ascending=False)
      .head(10)
)
muni_volume = (
    df.groupby("MUNICIPALITY", as_index=False)["VOLUME"]
      .sum()
      .sort_values("VOLUME", ascending=False)
      .head(10)
)

print("\nTop 10 municipalities by total VALUE:")
for _, row in muni_value.iterrows():
    print(f"- {row['MUNICIPALITY']}: {fmt(int(row['VALUE']))}")

print("\nTop 10 municipalities by total VOLUME:")
for _, row in muni_volume.iterrows():
    print(f"- {row['MUNICIPALITY']}: {fmt(int(row['VOLUME']))}")


There are 47 unique Drug Type Codes present in the data, with the 5 most used codes being: 1.  A10AB05 (10.051) 2.  A10BA02 (8.401) 3.  A10BK01 (7.218) 4.  A10BJ06 (7.079) 5.  A10BH01 (6.555)

There are 47 unique Drug Type Codes present in the dataset. The most common Drug Type Code in the dataset is A10AB05 (“None”), present in 10.051 rows/sales.  The least common Drug Type Code in the dataset is A10AE54 (“None”), present in 14 rows/sales.

There are 40 unique municipalities in the dataset.

Top 10 municipalities by total VALUE:
- København: 68.178.168
- Esbjerg: 22.178.539
- Vejle: 19.711.710
- Næstved: 19.062.031
- Sønderborg: 18.440.478
- Slagelse: 18.434.502
- Kolding: 18.367.565
- Horsens: 17.830.119
- Randers: 17.810.378
- Viborg: 17.535.757

Top 10 municipalities by total VOLUME:
- København: 233.895
- Esbjerg: 79.329
- Vejle: 68.412
- Horsens: 62.760
- Randers: 61.622
- Kolding: 60.502
- Slagelse: 59.816
- Næstved: 59.769
- Viborg: 58.584
- Sønderborg: 58.395
